# Rekurentne nevronske mreže

V prejšnjem modulu smo obravnavali bogate semantične predstavitve besedila. Arhitektura, ki smo jo uporabljali, zajema združeni pomen besed v stavku, vendar ne upošteva **vrstnega reda** besed, saj operacija združevanja, ki sledi vdelavam, odstrani to informacijo iz izvirnega besedila. Ker ti modeli ne morejo predstavljati vrstnega reda besed, ne morejo rešiti bolj zapletenih ali dvoumnih nalog, kot je generiranje besedila ali odgovarjanje na vprašanja.

Za zajem pomena zaporedja besedila bomo uporabili nevronsko mrežno arhitekturo, imenovano **rekurentna nevronska mreža** ali RNN. Pri uporabi RNN pošljemo naš stavek skozi mrežo eno po eno enoto (token), mreža pa ustvari neko **stanje**, ki ga nato skupaj z naslednjim tokenom ponovno vnesemo v mrežo.

![Slika prikazuje primer generiranja z rekurentno nevronsko mrežo.](../../../../../translated_images/sl/rnn.27f5c29c53d727b5.webp)

Glede na vhodno zaporedje tokenov $X_0,\dots,X_n$ RNN ustvari zaporedje blokov nevronske mreže, ki ga trenira od začetka do konca z uporabo povratne propagacije. Vsak blok mreže prejme par $(X_i,S_i)$ kot vhod in proizvede $S_{i+1}$ kot rezultat. Končno stanje $S_n$ ali izhod $Y_n$ gre v linearni klasifikator za ustvarjanje rezultata. Vsi bloki mreže imajo iste uteži in se usposabljajo od začetka do konca z eno potezo povratne propagacije.

> Slika zgoraj prikazuje rekurentno nevronsko mrežo v odvitih oblikah (na levi) in v bolj kompaktni rekurentni predstavitvi (na desni). Pomembno je razumeti, da imajo vse RNN celice iste **deljive uteži**.

Ker skozi mrežo potekajo vektorji stanj $S_0,\dots,S_n$, se RNN lahko nauči zaporednih odvisnosti med besedami. Na primer, ko se beseda *not* pojavi nekje v zaporedju, se lahko nauči zanikovati določene elemente znotraj vektorja stanja.

Vsaka RNN celica vsebuje dve utežni matriki: $W_H$ in $W_I$, ter pristranskost $b$. Pri vsakem RNN koraku, glede na vhod $X_i$ in vhodno stanje $S_i$, se izhodno stanje izračuna kot $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, kjer je $f$ aktivacijska funkcija (pogosto $\tanh$).

> Za probleme, kot je generiranje besedila (ki jih bomo obravnavali v naslednji enoti) ali strojni prevod, želimo tudi na vsakem RNN koraku dobiti neko izhodno vrednost. V tem primeru obstaja še ena matrika $W_O$, izhod pa se izračuna kot $Y_i=f(W_O\times S_i+b_O)$.

Poglejmo, kako nam lahko rekurentne nevronske mreže pomagajo pri klasifikaciji našega nabora novic.

> Za poskusno okolje moramo zagnati naslednjo celico, da zagotovimo, da je zahtevana knjižnica nameščena, podatki pa predpomnjeni. Če delate lokalno, lahko naslednjo celico preskočite.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Pri usposabljanju velikih modelov je lahko dodeljevanje pomnilnika na GPU-ju problematično. Prav tako bomo morda morali eksperimentirati z različnimi velikostmi minibatchev, da bodo podatki stali v naš GPU pomnilnik, a bo usposabljanje še vedno dovolj hitro. Če ta kodo poganjate na lastnem GPU računalniku, lahko eksperimentirate z nastavitvijo velikosti minibatcha za pospešitev usposabljanja.

> **Opomba**: Znano je, da določene različice NVidia gonilnikov po usposabljanju modela ne sprostijo pomnilnika. V tem zvezku poganjamo več primerov, kar lahko v nekaterih postavitvah povzroči izčrpanje pomnilnika, še posebej, če delate lastne eksperimente znotraj istega zvezka. Če naletite na nenavadne napake ob začetku usposabljanja modela, boste morda želeli znova zagnati jedro zvezka.


In [3]:
batch_size = 16
embed_size = 64

## Preprost RNN klasifikator

V primeru enostavnega RNN-jevska ponavljajoča enota je preprosta linearna mreža, ki prejme vhodni vektor in vektor stanja ter proizvede nov vektor stanja. V Kerasu je to mogoče predstaviti z `SimpleRNN` plastjo.

Čeprav lahko neposredno v RNN plast pošljemo enojno kodirane enote, to ni pametna ideja zaradi njihove visoke dimenzionalnosti. Zato bomo uporabili vdelano plast za znižanje dimenzionalnosti besednih vektorjev, nato RNN plast in na koncu `Dense` razvrščevalnik.

> **Opomba**: V primerih, ko dimenzionalnost ni tako visoka, na primer pri tokenizaciji na ravni znakov, je lahko smiselno neposredno poslati enojno kodirane enote v RNN celico.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Opomba:** Tu uporabljamo neneurjeno vdelano plast za preprostost, vendar za boljše rezultate lahko uporabimo vnaprej naučeno vdelano plast z uporabo Word2Vec, kot je opisano v prejšnji enoti. Dobro bi bilo, da prilagodite to kodo za delo z vnaprej naučenimi vdelavami.

Zdaj pa trenirajmo naš RNN. RNN-ji so na splošno precej težavni za treniranje, ker je, ko so RNN celice razpakirane po dolžini zaporedja, število plasti, vključenih v povratno propagacijo, zelo veliko. Zato moramo izbrati manjšo hitrost učenja in trenirati omrežje na večjem podatkovnem setu, da dosežemo dobre rezultate. To lahko traja kar dolgo, zato je uporaba GPU-ja priporočljiva.

Da pospešimo stvari, bomo RNN model trenirali le na naslovih novic, opustili bomo opis. Lahko poskusiš trenirati tudi z opisom in vidiš, če znaš usposobiti model.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Opomba**: natančnost je tukaj verjetno nižja, ker usposabljamo samo na naslovih novic.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Zdaj, ko uporabljamo maskiranje, lahko model treniramo na celotnem naboru podatkov naslovov in opisov.

> **Opomba**: Ste opazili, da smo uporabljali vektorizator, usposobljen na novičarskih naslovih, ne pa na celotnem besedilu članka? Potencialno lahko to povzroči, da se nekateri tokeni prezrejo, zato je bolje ponovno usposobiti vektorizator. Vendar pa ima to morda le zelo majhen vpliv, zato bomo zaradi enostavnosti ostali pri prej usposobljenem vektorizatorju.


## LSTM: Dolgoročno kratkoročni spomin

Eden glavnih problemov RNN-jev je **izginjanje gradientov**. RNN-ji so lahko precej dolgi in morda imajo težave s tem, da gradienti pridejo vse do prve plasti omrežja med povratno propagacijo. Ko se to zgodi, omrežje ne more naučiti povezav med oddaljenimi tokeni. Ena izmed možnosti za izogibanje temu problemu je uvedba **izrecnega upravljanja stanja** z uporabo **ključavnic**. Dve najpogostejši arhitekturi, ki uvajata ključavnice, sta **long short-term memory** (LSTM) in **gated relay unit** (GRU). Tukaj bomo obravnavali LSTM-je.

![Slika prikazuje primer dolgo kratkoročnega spominskega elementa](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Omrežje LSTM je organizirano podobno kot RNN, vendar se iz plasti v plast prenašata dve stanji: dejansko stanje $c$ in skriti vektor $h$. Pri vsaki enoti se skriti vektor $h_{t-1}$ združi z vhodom $x_t$ in skupaj nadzirata, kaj se zgodi s stanjem $c_t$ in izhodom $h_{t}$ preko **ključavnic**. Vsaka ključavnica ima sigmoidalno aktivacijo (izhod v območju $[0,1]$), ki jo lahko razumemo kot bitno masko, ko se pomnoži z vektorjem stanja. LSTM-ji imajo naslednje ključavnice (od leve proti desni na zgornji sliki):
* **ključavnica za pozabljanje**, ki določi, katere komponente vektorja $c_{t-1}$ moramo pozabiti in katere prenesti naprej. 
* **ključavnica za vhod**, ki določi, koliko informacij iz vhodnega vektorja in prejšnjega skritega vektorja naj se vključi v vektor stanja.
* **ključavnica za izhod**, ki vzame nov vektor stanja in odloči, katere njegove komponente se bodo uporabile za izdelavo novega skritega vektorja $h_t$.

Komponente stanja $c$ lahko razumemo kot zastavice, ki se lahko vklapljajo in izklapljajo. Na primer, ko naletimo na ime *Alice* v zaporedju, ugotovimo, da se nanaša na žensko in v stanju dvignemo zastavico, ki pravi, da imamo v stavku samostalnik ženskega spola. Ko nato naletimo na besede *and Tom*, dvignemo zastavico, ki pravi, da imamo množinski samostalnik. S takim upravljanjem stanja lahko torej sledimo slovničnim lastnostim stavka.

> **Opomba**: Tukaj je odličen vir za razumevanje notranjosti LSTM-jev: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) avtorja Christopher Olah.

Čeprav notranja struktura LSTM-celi lahko izgleda kompleksno, Keras to implementacijo skriva v sloju `LSTM`, zato je edina stvar, ki jo moramo narediti v zgornjem primeru, zamenjati rekurentni sloj:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Upoštevajte**, da je učenje LSTM-jev prav tako precej počasno in morda sprva ne boste opazili velikega povečanja natančnosti. Morda boste morali nadaljevati z učenjem še nekaj časa, da dosežete dobro natančnost.


## Dvosmerni in večplastni RNN-ji

V dosedanjih primerih delujejo rekurentne mreže od začetka zaporedja do konca. To se nam zdi naravno, saj sledi isti smeri, kot v kateri beremo ali poslušamo govor. Vendar pa za scenarije, ki zahtevajo naključni dostop do vhodnega zaporedja, je smiselno izvajati rekurentne izračune v obeh smereh. RNN-ji, ki dovoljujejo izračune v obeh smereh, se imenujejo **dvosmerni** RNN-ji, ki jih lahko ustvarimo tako, da ovojnemo rekurentni sloj s posebnim slojem `Bidirectonal`.

> **Opomba**: Sloj `Bidirectional` naredi dve kopiji sloja znotraj sebe in nastavi lastnost `go_backwards` ene od teh kopij na `True`, kar povzroči, da poteka v nasprotni smeri po zaporedju.

Rekurentne mreže, enosmerne ali dvosmerne, zajamejo vzorce znotraj zaporedja in jih shranijo v stanje ali pa jih vrnejo kot izhod. Tako kot pri konvolucijskih mrežah lahko za zajem višjenivojskih vzorcev, sestavljenih iz nižjenivojskih vzorcev, pridobljenih s prvim slojem, zgradimo še en rekurentni sloj za prvim. To nas pripelje do pojma **večplastni RNN**, ki je sestavljen iz dveh ali več rekurentnih mrež, kjer je izhod prejšnjega sloja podan naslednjemu kot vhod.

![Slika prikazuje večplastno dolgoročno-krajnoročno RNN RNN](../../../../../translated_images/sl/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Slika iz [te čudovite objave](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) avtorja Fernanda Lópeza.*

Keras olajša sestavljanje teh mrež, saj je potrebno le dodati več rekurzivnih plasti modelu. Za vse plasti razen zadnje moramo določiti parameter `return_sequences=True`, ker potrebujemo, da plast vrne vse vmesne stanje, ne samo končno stanje ponavljajoče se operacije.

Zgradimo dvoplastni bidirekcijski LSTM za naš problem klasifikacije.

> **Opomba** ta koda zopet traja precej dolgo, da se izvede, vendar nam daje najvišjo natančnost, ki smo jo doslej videli. Zato je morda vredno počakati in videti rezultat.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN za druge naloge

Do sedaj smo se osredotočili na uporabo RNN za razvrščanje tekstovnih zaporedij. Vendar pa lahko opravijo mnogo več nalog, kot so generiranje besedila in strojno prevajanje &mdash; ti nalogi bomo obravnavali v naslednji enoti.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
